# Experiments Dashboard

This notebook reads the experiment log produced by `EDA.ipynb` and builds slide-friendly tables/plots.

**Where to look**
- Log file: `artifacts/experiments.jsonl`
- Table export: `artifacts/experiments.csv`

Workflow:
1) Run experiments in `EDA.ipynb`.
2) Make sure the last cell "Save experiment results" ran.
3) Open this notebook and `Run All`.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

LOG_PATH = 'artifacts/experiments.jsonl'


In [ ]:
def read_jsonl(path):
    rows = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    except FileNotFoundError:
        pass
    return rows

rows = read_jsonl(LOG_PATH)
print('runs:', len(rows))


In [ ]:
df = pd.DataFrame(rows)
if df.empty:
    df
else:
    # Put newest on top
    if 'logged_at_utc' in df.columns:
        df = df.sort_values('logged_at_utc', ascending=False)

    # Slide-friendly view
    cols = [c for c in [
        'exp_id','exp_name','model_name','n_samples','split_train','split_val','split_test',
        'stage1_test_acc','stage2_test_acc','baseline_test_acc','notes','logged_at_utc'
    ] if c in df.columns]
    display(df[cols])


## Comparison plot (test accuracy)

This plot helps you pick the best configurations for your 5-minute presentation.


In [ ]:
if not df.empty:
    # Create a label per run
    label_col = 'exp_name' if 'exp_name' in df.columns else 'exp_id'
    y_cols = [c for c in ['stage2_test_acc','stage1_test_acc','baseline_test_acc'] if c in df.columns]

    plt.figure(figsize=(10, 4))
    for col in y_cols:
        sub = df[[label_col, col]].dropna()
        if sub.empty:
            continue
        plt.plot(sub[label_col], sub[col], marker='o', label=col)

    plt.xticks(rotation=30, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Test accuracy')
    plt.title('Test accuracy by experiment')
    plt.legend()
    plt.tight_layout()
    plt.show()
